# Design of Experiments (DOE)

This notebook demonstates use of Design Of Experiment (DOE) capabilities included in the ProcessOptimizer code.

First, we define the search space that we are interest in like we would for a normal optimization problem.
We do this using the Space class from the ProcessOptimizer library.

Note that we have also included a 2-level categorical variable.
At current, the version of D-optimal designs in ProcessOptimizer does not support categorical variables with more than two levels.

In [2]:
from ProcessOptimizer.space import Categorical, Integer, Real, Space
import numpy as np

factor_space = Space(dimensions=[
    Real(10, 40, name='var_x1'),
    Integer(20, 100, name='var_x2'),
    Integer(-30, 30, name='var_x3'),
    Categorical(['a', 'b'], name='var_x4')
    ])

## D-optimal design

D-optimal design is a type of experimental design that is used to find the optimal threatment combinations for a given number of experiments.

Besides the search space, we also need to define the number of experiments that we want to run and the type of model we want to be able to fit to the data.

In [6]:
from ProcessOptimizer.doe import get_optimal_DOE

number_of_experiments = 12

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments)

print("Factor names:")
print(factor_names)
print("Design:")
print(design)

order is 2
Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[[np.float64(40.0) np.int64(100) np.int64(-30) 'b']
 [np.float64(10.0) np.int64(100) np.int64(-30) 'a']
 [np.float64(40.0) np.int64(100) np.int64(30) 'b']
 [np.float64(10.0) np.int64(100) np.int64(30) 'b']
 [np.float64(40.0) np.int64(100) np.int64(30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(-30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(-30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(30) 'a']
 [np.float64(40.0) np.int64(20) np.int64(-30) 'b']
 [np.float64(40.0) np.int64(100) np.int64(-30) 'a']
 [np.float64(40.0) np.int64(20) np.int64(30) 'a']]


This is consistent with the type of each dimension in the search space, but not particularly printer friendly. Thus we convert the numbers to strings and get:

In [7]:
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['40.0' '100' '-30' 'b']
 ['10.0' '100' '-30' 'a']
 ['40.0' '100' '30' 'b']
 ['10.0' '100' '30' 'b']
 ['40.0' '100' '30' 'a']
 ['10.0' '20' '-30' 'b']
 ['40.0' '20' '30' 'b']
 ['40.0' '20' '-30' 'a']
 ['10.0' '20' '30' 'a']
 ['40.0' '20' '-30' 'b']
 ['40.0' '100' '-30' 'a']
 ['40.0' '20' '30' 'a']]


Besides the design space and number of experiments, a number of other parameters can be set. These are:
- `design_type` : Various keywords can be used to specify the regression model that the design should be optimal for.
- `model` : As an alternative to `design_type`, a model can be specified directly. `design_type` and `model` are mutually exclusive. If both are specified, `design_type` will be used.
- `replicates` : A number of replicates can be specified. This is useful when the same experiment is to be run multiple times.
- `sorting` :  Various keywords can be used to specify sorting of experiments in the design.
- `res` : A resolution can be specified. This controls how coarsely the factors should be sampled during the design optimization. 

`design_type` can be set to one of the following:
- `linear` : Simple linear regression model without interaction terms.
- `screening` : Screening design with main effects and two-factor interactions.
- `response` : Response surface design with main effects, two-factor interactions, and quadratic effects for non-categorical factors.
- `optimization` : Optimization design with main effects, two- and three-factor interactions, and quadratic and cubic effects for non-categorical factors.

In [2]:
from ProcessOptimizer.doe import get_optimal_DOE
from ProcessOptimizer.space import Integer, Real, Space

You give generator a Space object generated by the `ProcessOptimizer` Space class.

ALSO INFORMATION ABOUT HTE OTHER PARAMETERS

In [4]:
factor_space = Space(dimensions=[Real(10, 40, name='ul_indicator'),
                                     Integer(20, 100, name='ul_base'),
                                     Integer(20, 100, name='ul_acid'),
                                     ])

design, factor_names = get_optimal_DOE(factor_space, 10, design_type='response')

In [5]:
print(design)
print(factor_names)

[[20.90909090909091, 100, 100], [10.0, 20, 100], [10.0, 20, 20], [40.0, 100, 49], [29.09090909090909, 71, 20], [40.0, 42, 100], [40.0, 20, 20], [10.0, 100, 20], [26.363636363636363, 20, 64], [10.0, 71, 71]]
['ul_indicator', 'ul_base', 'ul_acid']


NOTE categorical variables are not yet supported.

In [6]:
factor_names = factor_space.names
my_model1 = f'{factor_names[0]} + {factor_names[1]} + {factor_names[2]} + {factor_names[0]}:{factor_names[1]} + {factor_names[0]}:{factor_names[2]} + {factor_names[1]}:{factor_names[2]}'
print(my_model1)

ul_indicator + ul_base + ul_acid + ul_indicator:ul_base + ul_indicator:ul_acid + ul_base:ul_acid


In [8]:
get_optimal_DOE(factor_space, 7, model=my_model1)

([[10.0, 100, 20],
  [10.0, 100, 100],
  [40.0, 20, 100],
  [40.0, 100, 100],
  [10.0, 20, 100],
  [40.0, 100, 20],
  [10.0, 20, 20]],
 ['ul_indicator', 'ul_base', 'ul_acid'])

# OLD BELOW

In [12]:
my_model2 = f'{factor_names[0]} + {factor_names[1]} + {factor_names[2]} + pow({factor_names[1]}, 2)'
print(my_model2)

ul_indicator + ul_base + ul_acid + pow(ul_base, 2)


In [1]:
get_optimal_DOE(factor_space, 3, model=my_model2)

NameError: name 'get_optimal_DOE' is not defined

In [ ]:
get_optimal_DOE(factor_space, 5, model=my_model2)